# HW10: Exploring LLMs

In this homework assignment, we will walk you through how to use and finetune a variety of LLMs with web, API, and locally hosted tools.

Specifically, we will explore:
- `Mistral-7B-v0.1`, a small-ish neural language model developed by Mistral AI and [released in September 2023](https://huggingface.co/mistralai/Mistral-7B-v0.1)
- `Llama-3.1-8B-Instruct`, a small-ish neural language model developed by Meta and [released in July 2024](https://huggingface.co/meta-llama/Llama-3.1-8B)
- `Gemma-2-2B-it`, a small neural language model developed by Google and [released in July 2024](https://huggingface.co/google/gemma-2-2b-it)

The first two sections of this assignment should be completable on any computer with Internet access, but **we recommend that you complete the last (model finetuning) section on Google CoLab with T4 GPU, or any other server with a modern GPU and at least 16GB GPU RAM**.

If you are doing this assignment on Google CoLab, we suggest using a CPU runtime environment for all earlier sections, and switching to a T4 GPU runtime when you start model finetuning. Note that changing runtime environments will delete all imports, functions, and other variables held in memory, so you may need to re-run some cells. We recommend that you *do not* delete old cell outputs when you change runtime environments.

Based on course staff testing, we estimate that this assignment will take at the very least 20 minutes on CPU, and 45 minutes on T4 GPU. (It will likely take maybe 2x longer at least due to debugging and hyperparameter exploration. Start early and keep careful notes on what you have tried so far.)

---

Some context: You likely recognize Meta and Google as companies already. Mistral AI is a French AI startup, also known for training and releasing multiple well-recognized open-weight large language models. [Mistral 7B](https://huggingface.co/mistralai/Mistral-7B-v0.1) and its associated family of models was released as an open-weight model (meaning anyone can download all the model weights and train/use it as they like) in 2023, with [claims of outperforming Meta's Llama 2 13B (another famous effectively-open-weight model at the time) on notable benchmarks](https://mistral.ai/news/announcing-mistral-7b). Of course, there have been many new models released since all three of the ones we are playing with in this assignment.

The "7B", "8B", and "2B" in these model names denote how many parameters are in the neural network - so there are 7 billion parameters in Mistral 7B.
The "Instruct" or "it" means that the base language model was then finetuned with a special chat or instruction-following format.
The "v0.1", "3.1", or "2" indicate which generation within a model family it is, so "Gemma-2" was in the second group of Gemma model releases.
These naming conventions aren't strictly followed, but open-weight model names tend to use the "name/version, size, instruct?" format.

While all three of these models are *very very large* compared to all other models we have examined in lecture or implemented/used in homework assignments so far, and larger than some other modern LLMs such as Hugging Face's [SmolLM 2 135M](https://huggingface.co/HuggingFaceTB/SmolLM2-135M), they are still *very very small* compared to several well-known chat-interface-supporting models such as OpenAI's [GPT-3](https://en.wikipedia.org/wiki/GPT-3) (175B parameters), OpenAI's [GPT-4.1](https://platform.openai.com/docs/models/gpt-4.1) (speculated around 1.7T parameters, possibly a "Mixture of Experts" - which means not all components of the model are always used at inference time), DeepSeek's [DeepSeek-V4-Pro](https://huggingface.co/deepseek-ai/DeepSeek-V4-Pro) (1.6T MoE parameters; 49B activated parameters), or Anthropic's [Claude Opus 4.6](https://www.anthropic.com/news/claude-opus-4-6) (speculated around 1-2T parameters, probably something MoE).

---

**We will explore the following topics**:
- *Prompt completions*. You should observe that the quality of text generated can be high-quality, but not necessarily factually accurate.
- *Probabilities*. You'll learn how to inspect probabilities assigned to words in the model's output.
- *Baseline vs instruct*. You'll observe the behavioral and usage differences between baseline vs instruction-tuned models.
- *Few-shot learning*. We share an example of few-shot learning and have you try it on the model.
- *Zero-shot learning*. We will explore the zero-shot capabilities of a pre-trained LLM, and you will design zero-shot prompts for four tasks:
  1. Summarization
  2. Question-Answering
  3. Simplification
  4. Translation
- *Model fine-tuning*. You will learn how to fine-tune a LLM to take a [Wikipedia infobox](https://en.wikipedia.org/wiki/Help:Infobox) as input and generate some biography text as output. You'll also write your own code to do the reverse - given a biography, extract the attributes and values into the style of a Wikipedia infobox.

**We will also be using our LLM via three different methods**:
- *Online playgrounds*. There are multiple free (or low-cost) tools for generating LLM text completions online.
  - This can be very useful for prototyping or getting familiar with the behavior of new models or prompts, by giving users control over model generation settings.
  - While services such as ChatGPT, DeepSeek, Claude, etc. could arguably be considered a type of online playground, they are usually simultaneously more restrictive (using chat-only formats, not giving users control over generation settings, applying filters on model input and/or output, not showing plaintext information about what is generated) and more flexible (the model or tool may allow external tool integrations or multimodal media more than a single underlying LLM could support).
- *API calls*. These are increasingly popular, especially for scenarios where developers do not want to or cannot keep a LLM on a local server, or do not have access to a local GPU to perform inference.
  - This is often the only usable interface for developers to use closed-weight LLMs such as OpenAI or Anthropic models. They are also fairly useful for serverless projects. In addition, API providers may be able to make things more efficient without you needing to do very much work. For example, they may be able to cache common requests or automatically handle large batches of requests as appropriate.
- *Running a local model*. For open-weight or open-source models such as [Mistral](https://huggingface.co/mistralai/models), [Llama](https://huggingface.co/meta-llama/collections), [Gemma](https://huggingface.co/collections/google/gemma-3-release-67c6c6f89c4f76621268bb6d), or [OLMo](https://huggingface.co/collections/allenai/olmo-2-674117b93ab84e98afc72edc) model families, we can download the model weights directly and use libraries such as [HuggingFace's Transformers library](https://github.com/huggingface/transformers) to run these on our own machines (or borrowed/rented servers), as long as we meet the appropriate system requirements.
  - This may be the only feasible option if you need to use LLMs for tasks involving sensitive information, are doing research involving model weights or architecture, or otherwise need to work with models at a finer level of detail.

Before you get started, please sign up for:
- [Nvidia NIM](https://build.nvidia.com/models?modal=signin): used for API access.
  - Note that as of July 2025, Nvidia NIM requires a phone number for verification (no VoIP numbers).
- [HuggingFace](https://huggingface.co/): used for downloading model weights and uploading your own finetuned model.
  - Note that Gemma models require you to share your HuggingFace account contact information in order to gain access permissions.
- [Weights and Biases](https://wandb.ai/): used for keeping track of model training and performance. (This can be opted out of, if you disable wandb logging in model finetuning.)

In order to complete this assignment, you will need working API keys at least the first two services.

Please let the course staff know if you encounter any issues with account registration, model access, or GPU runtime access.

# Prompt Completions

As a warm-up, we'll have you play with the [TextSynth online LLM playground](https://textsynth.com/completion.html). Note that TextSynth offers playground access for Mistral 7B (not the instruct version), Llama 3.1 8B Instruct, and 2 other larger instruct models. The free online service heavily limits how many tokens it generates, but it should produce enough tokens for our purposes.

First, let's learn some basic terminology:
- Prompt - the input to the model
- Completion - what the model outputs

There are also several settings on the playground. These are:
- `Model` - Which model we are using. For this example, we'll use "Mistral 7B".
- `top-k` - controls how the model generates. A model selects the next token from the top-k most likely ones.
- `top-p` - controls how the model generates. A model selects the next token among the most probable ones, so that their cumulative probability is larger than top-p.
- `temperature` - controls how the model generates. Higher temperature means the model tends to select less common tokens, which means higher diversity but potentially less relevant output.
- `max tokens` - the maximum number of tokens to generate
- `stop at` - you can specify what literal string output should cause the model to stop generating

Try inputting this prompt while using the default settings (`Model`="Mistral 7B", `top-k`=40, `top-p`=0.9, `temperature`=1.0, `max tokens`=200, `stop at`=empty):

> One of my favorite professors at the University of Pennsylvania is named

And then click "Complete Text" (or "Generate Another" if you already have a prompt from earlier) to generate a completion.

Copy and paste the full generated text below, including your prompt.

You might notice that the output often ends mid-sentence. TextSynth is configured to have a model generate text until it outputs the number of tokens specified by the `max tokens` variable, or until it produces the string specified by the `stop at` variable. You can click "More" to have the model continue generation. (TextSynth also offers some API access, and you can view documentation for additional arguments here: [https://textsynth.com/documentation.html#completions](https://textsynth.com/documentation.html#completions))

In [1]:
favorite_professor_completion = """
One of my favorite professors at the University of Pennsylvania is named Dani Rodrik, an economist who understands well that the global economy doesn’t always work out so well for those of us at the bottom, but who argues for the importance of trying to make it work all the same.

Rodrik is also an MIT professor and a former Harvard professor, and he writes an excellent blog on the blog aggregator site “Vox.” This week he had a post entitled “Is the world getting less fair?,” and he summarizes the findings of some very depressing research on global inequality, all of which can be found at a paper called “Dangerous Inequalities: the Political Consequences of Rising Global Income Inequality.”

Here’s Rodrik’s summary of the findings:

> The first finding is that the bottom has been falling out for developing countries, especially those that are commodity exporters, who are at risk of falling behind the technological frontier (
"""

Let's revise the prompt a little bit to encourage the model to produce more outputs about computer science professors.

Do the same thing as above, but now with this prompt:

> One of my favorite professors in computer science at the University of Pennsylvania is named

In [2]:
favorite_cs_professors_completion_1 = """
One of my favorite professors in computer science at the University of Pennsylvania is named Chris Jaynes. He has a very unconventional teaching style.

First of all, he's one of the youngest professors there and teaches two of the most popular classes on campus. He regularly gets 250-300 students in each of his courses.

He is unapologetically a great teacher. I have rarely seen him not wearing a tie, even in his casual attire. He's very meticulous and organized.

He doesn't have any formal credentials or fancy degrees. He started his own company and now sells his computer programs to universities.

He teaches like a college student — which is not a surprise, since he was just a few years older than many of his students. He even wears shorts.

One time, a student asked him why he was wearing shorts. He replied, "Because it's summer."

Another student asked him if he was a
"""

Mistral 7B can generate fluent text, but are not always grounded in fact. Let's do an online search for the professor that the LLM generated information about and check:

- Are they actually a CS professor?
- If they are a professor, where do they work? (If they're not a CS professor, feel free to write "not CS" or "not a professor".)

In [3]:
# What is the professor's name
cs_professor_name_1 = "Chris Jaynes"

# Do an online search and answer these questions:

# Is this person an actual CS professor somewhere?
actually_a_cs_professor_1 = False
# Where do they work?
institution_1 = "not CS"

When generating text completions, a LLM samples each new word/token according to its probability distribution. It draws each word at random in proportion to its propability. That randomness means that it can generate different completions. You can re-generate and get different completions each time.

Let's generate another 4 completions for the "favorite CS professor" prompt

> One of my favorite professors in computer science at the University of Pennsylvania is named

and do web searches to verify their accuracy.

*Tip: You can quickly generate another response with the "Generate Another" button.*

In [4]:
favorite_cs_professor_completion_2 = """
One of my favorite professors in computer science at the University of Pennsylvania is named Vasant Dhar. He has a very interesting course called the Art and Science of Happiness that is a mix of psychology, art, economics, and some of the more interesting philosophical concepts in Buddhism, Christianity, and Hinduism.

The main question he answers in his course is how to live a happy and fulfilled life. There’s no one answer to that question, but when you start applying the concepts to your own life, you get a new perspective that helps you better understand the choices you’ve made, and you can make more informed decisions going forward.

When you ask people what’s most important in their lives, the answers are all over the map.

- Happiness
- Family
- Money
- Health
- Love
- Career
- Personal Growth

These are all important and can have very different meanings to different people. But they all have one thing in common. It’s difficult
"""

favorite_cs_professor_completion_3 = """
One of my favorite professors in computer science at the University of Pennsylvania is named Dan Grossman. In an online class that he teaches, he has had me doing proofs and reasoning since day one.

“This guy doesn’t hold back at all,” I thought to myself in the beginning. I’m not taking this course to learn how to think in terms of mathematical theorems. I’m not taking this course to learn how to make arguments based on pure logic. That stuff is for high school.

It’s been about two weeks since my computer science course began and I’ve been proven wrong. It turns out that Dan was right — thinking is more than a natural skill. Learning how to think takes a lot of practice. Just because we’re able to follow along with Dan’s logic and reason doesn’t mean that we’re thinking in the right way. It takes practice to get better.

So, here’s the best way that I’ve learned to improve your thinking ability.
"""

favorite_cs_professor_completion_4 = """
One of my favorite professors in computer science at the University of Pennsylvania is named Mark Stefik.  He introduced me to the idea of “lifted inference”.  As the name implies, the concept of lifted inference comes from math.  In lifted inference, you take a math problem and solve it at a higher level.  For example, when doing arithmetic, you take a problem like “1+3” and solve it at a higher level by generalizing to “2+n” which is easier to solve.  This works for certain classes of problems, but there is a danger that if you raise the problem to a high enough level, you introduce uncertainty, for example, “a+b” where you have no way of knowing if a is greater or less than b.

I saw Mark again at the Stanford School of Education this year and I asked him if he was still using the lifted inference concept.  He said yes and he uses it for making better decisions.  He uses several examples:
"""

favorite_cs_professor_completion_5 = """
Daniel Krob. He taught two of my favorite courses: the first class was titled "Computer Architecture for the Software Engineer," and the second was titled "Multimedia." I distinctly remember how Daniel would often take the stage at the beginning of his lecture, without much ado, and simply state the learning objectives of the course as slides shown in Fig. 2.1. I thought that was a nice introduction to the course! But one of the things I remember vividly was that in both courses, we had weekly labs in which we were supposed to write a piece of code that demonstrated what we had just learned in the lecture. The labs were graded individually, but the code was also graded by every student of the class: We would simply check each other's code, and we would rate it from 0 to 10. The code with the best rating was given extra points, the second-best code got one point fewer, and
"""

# Do an online search for these professors

cs_professor_name_2 = ""
actually_a_cs_professor_2 = False
institution_2 = "not CS"

cs_professor_name_3 = ""
actually_a_cs_professor_3 = False
institution_3 = "not CS"

cs_professor_name_4 = ""
actually_a_cs_professor_4 = False
institution_4 = "not CS"

cs_professor_name_5 = ""
actually_a_cs_professor_5 = False
institution_5 = "not CS"

Optional: Feel free to try out the other larger instruct models available through TextSynth, or tinker with the generation settings or prompt.
- Which of these models are more accurate about CS professors?
- Do any particular generation settings produce more accurate texts?
- Do any of these models tend to produce differently formatted texts?
- What happens if you ask the model to complete a prompt such as "my favorite course at UPenn is CS" without whitespace at the end?

Feel free to leave any reflections from this exploration in a note below.

In [5]:
"""
[Optional: Leave some notes about anything particularly interesting you learned from this section here]
"""

'\n[Optional: Leave some notes about anything particularly interesting you learned from this section here]\n'

# Few-Shot Learning

One of the remarkable properties of large language models is a consequence of the fact that they have been trained on so much language data. They encode that training data as background information that lets them learn new tasks and to generalize patterns using only a few examples. This is called "few-shot learning".

Here's an example. Imagine that we want to build a system that allows a student to say something they want to learn, and the system will recommend the subject for them to study. Here are some example inputs and outputs to our program:

```
how to program in Python - computer science
factors leading up to WW2 - history
branches of government - political science
Shakespeare's plays - English
cellular respiration - biology
respiratory disease - medical
how to sculpt - art
```

We can use these 7 examples (and possibly fewer!) as a prompt, and it will function as few shot learning by figuring out what our pattern is, and being able to emulate this pattern to perform the task for new inputs.

Try pasting those 7 earlier examples into the playground, and then appending a lesson after the examples (one lesson at a time) to see what the output is.

```
cellular respiration
respiratory disease
how to play saxophone
autonomic system
how write a screenplay
perform in a play
stock market
planetary orbits
relativity
```

Fill in the dictionary below by using the playground with Mistral 7B. Replace the TODOs with the subjects that the model predicts.

In [6]:
few_shot_subject_classification_results = {
    "cellular respiration" : "biology",
    "respiratory disease" : "biology",
    "how to play saxophone" : "instrumental music",
    "autonomic system" : "anatomy",
    "how write a screenplay" : "film studies",
    "perform in a play" : "drama",
    "stock market" : "business",
    "planetary orbits" : "astronomy",
    "relativity" : "quantum mechanics - physics",
}

What were the exact prompt and playground settings that you used to predict the subject for "planetary orbits"? What was the full model output produced?

In [7]:
few_shot_subject_classification_prompt = """
What are the subjects that relate to these topics?
"""
few_shot_subject_classification_settings = """
Model: Mistral 7B, top-K: 40, top-p: 0.9, temperature: 1.0, max tokens: 200, stop at: empty
"""
few_shot_subject_classification_output = """
how to program in Python - computer science
factors leading up to WW2 - history
branches of government - political science
Shakespeare's plays - English
cellular respiration - biology
respiratory disease - medical
how to sculpt - art

relativity, quantum mechanics - physics
how to program in Java - computer science
solar systems - astronomy

In the US, there are plenty of free sources to learn about any of the topics above. You can find videos on YouTube, read books from your local library, find open educational courses online (some of which come from top universities)... the list is endless.

It's your responsibility to learn. If you don't, it's nobody's fault but your own.
User 10: It is not about how much we know. Some of us know quite a lot. My point is that we don't work in a job that is directly relevant to our studies. I'm lucky if I'm able to do some data analysis. And even that is done using excel or the company's database.

And as I said. I'm not going to invest any more time to learn stuff if it is not
"""

Optional: Let's consider how using this as part of a larger tool might work.
- What were some of the challenges that you ran into while setting up an appropriate prompt for each lesson?
- How might you automate this prompt creation or post-process the output?
- If you had greater control over the LLM inference process than the playground gives you, what are some ways you could improve this prediction process?

Feel free to leave any reflections from this exploration in a note below.

In [8]:
"""
[Optional: Leave some notes about anything particularly interesting you learned from this section here]
"""

'\n[Optional: Leave some notes about anything particularly interesting you learned from this section here]\n'

## Using API Calls

Let's take a look at how to use a LLM via an API provider now, so that we don't have to manually enter inputs into the browser.

We will use the Nvidia NIM APIs. Specifically, we will use [Llama-3.1-8B-Instruct](https://build.nvidia.com/meta/llama-3_1-8b-instruct). You can test the model in the "Playground" web interface (designed for strictly chat-formatted inputs).

If you look at the "Build" page, you'll see a sample of code for whatever prompt you have currently entered. For example, if we type "What is the best course in computer science at the University of Pennsylvania?" into the Playground chat box, then it gives us this example code:

```python
from openai import OpenAI

client = OpenAI(
    base_url = "https://integrate.api.nvidia.com/v1",
    api_key = "$NVIDIA_API_KEY"
)

completion = client.chat.completions.create(
    model="meta/llama-3.1-8b-instruct",
    messages=[{"content":"What is the best course in computer science at the University of Pennsylvania?","role":"user"}],
    temperature=0.2,
    top_p=0.7,
    max_tokens=1024,
    stream=True
)

for chunk in completion:
    if chunk.choices and chunk.choices[0].delta.content is not None:
        print(chunk.choices[0].delta.content, end="")
```

This is Python code, so it will be fairly straightforward for us to use it as a starting point. We can reference the [OpenAI API documentation for chat completion](https://platform.openai.com/docs/api-reference/chat/create) to make changes as appropriate, although the Nvidia API does not necessarily support all requests the same way that OpenAI would.

First, we'll need to install the [OpenAI API library](https://platform.openai.com/docs/api-reference/introduction) via `pip`. (While we are using the Nvidia NIM APIs, OpenAI effectively set a standard for modern LLM APIs, so most LLM APIs will work with this package or something extremely similar by default.)

You can use Unix commands in a CoLab notebook by prefixing them with an exclamation point. The example below uses a `%` prefix instead — this guarantees that the installation is done using the same Python version that your notebook is using. Change line 2 from `%pip...` to `!pip...` if you're running into trouble. (The `%%capture` command before that just supresses the output of running the Unix command.  You can remove it if you want to see the progress of the command).

In [9]:
%%capture
%pip install openai

Next, you will enter your secret key for the Nvidia NIM API. You can generate your API key [here](https://build.nvidia.com/settings/api-keys).  

We will enter it as a password, so that the raw text of it doesn't get saved in your Python notebook. That would be bad in the case that you upload or share your notebook, because then other people could use your key.

Even though we are only using the Nvidia NIM free services for this homework assignment, someone could "borrow" your key in case you kept your account for future paid usage. Excess usage might also result in API keys being disabled or otherwise make the service more prone to abuse. (It is good to be careful about API key security in general, since most other API services would definitely bill you for any and all usage, or cause more severe privacy issues!)

In [10]:
from getpass import getpass
import os

print('Enter Nvidia API key:')
os.environ['NVIDIA_API_KEY'] = getpass().strip()

Enter Nvidia API key: nvapi-REDACTED
··········


Earlier, the TextSynth playground focused on *text* completions - that is, given an input text, produce some following text that is coherent and makes sense given the earlier context. Note that it always focused on text completions regardless of whether the model was trained to recognize instructions or not.

Now, we are using an instruct model with an API and the OpenAI package designed to focus on *chat* completions - that is, given an input that hopefully contains some system instructions and a previous chat history, the model produces some following text that can be understood as a response chat message. Note that this still fundamentally does the same thing as a text completion. The goal is still to produce reasonable and realistic text, but now there's new formatting constraints that are relevant to model behavior.

(You may have noticed some funny tokens if you used the instruct models to do text completions on TextSynth earlier. That's related to the way that instruct models are finetuned! Under the hood, instruct models are generally just base models that went through extra training with some special format datasets.)

While we could do text completions using the Nvidia NIM API and the OpenAI package, they really aren't designed for that usage. We will briefly revisit text completions later when using local models.

In the meantime, let's write a function that tries to emulate our few-shot learning task earlier, taking a topic as input and outputting a subject to study if you want to learn about that topic.

In [11]:
import openai
import os
import time

def generate_subject_few_shot(topic):
    few_shot_prompt = """how to program in Python - computer science
factors leading up to WW2 - history
branches of government - political science
Shakespeare's plays - English
cellular respiration - biology
respiratory disease - medical
how to sculpt - art
"""

    # All LLM completions are generated by the chat (instruct) model now.
    # Llama-3.1-8B-Instruct expects a certain chat message format:
    # starting with an optional "system" message, then alternating between
    # "user" messages to denote human input and "assistant" messages to denote model responses
    # It works best when it's given a "system" message with baseline instructions about how the model should behave.
    # We'll use this one:
    system_message = {
        "role" : "system",
        "content" : f"""You are a helpful assistant. Your task is to respond with a message that appropriately matches the user's input.

Here are some examples in the format \"user input - result\".

{few_shot_prompt}

Do not provide explanations or notes.
""",
    }

    client = openai.OpenAI(
        base_url = "https://integrate.api.nvidia.com/v1",
        api_key = os.environ['NVIDIA_API_KEY'],
    )
    response = client.chat.completions.create(
        model="meta/llama-3.1-8b-instruct",
        messages= [
            system_message,
            {"role" : "user", "content" : topic + " - "}, # simulate a user prompt
        ],
        temperature=0.7,
        max_tokens=256,
        top_p=1,
        frequency_penalty=0,
        presence_penalty=0,
        stop=["\n"],
    )
    # I recommend putting a short wait after each call,
    # since there is typically a rate limit for API platforms.
    # A reasonable initial wait limit is something like 60 requests/min for new accounts.
    # (On many platforms, accounts that are older and/or have paid for more usage will have this cap raised over time.)
    time.sleep(1)

    # the response from the API is a JSON object that contains
    # the completion to your prompt plus some other information.  Here's how to access
    # just the text of the completion.
    return response.choices[0].message.content.strip()


topic = "cellular respiration"
generate_subject_few_shot(topic)

'biology'

That's it! That's an example of how to write a function call to OpenAI-shaped APIs in order to configure an LLM to output a subject for a topic.

Here is some information about the different arguments that we use in the `openai.Completion.create` call:
- `model` - This specifies which model to use, depending on what options the API provider makes available.
  - There are a wide array of models available on [Nvidia NIM](https://build.nvidia.com/models).
  - We can use the OpenAI package with other API providers (requires changing `base_url` and `api_key` args when creating our client object), giving us a [wide]((https://platform.claude.com/docs/en/about-claude/pricing) [variety](https://api-docs.deepseek.com/) [of](https://developers.openai.com/api/docs/pricing) [models usable](https://huggingface.co/docs/inference-providers/main/en/index?python-clients=openai#python), each with different pricing and sometimes different argument requirements.
- `messages` - this is the list of messages that guide how the model should respond.
  - The first message is a system prompt — an overall setting that specifies how the model should behave. The second message is the user prompt - something analogous to a chat message from the user.
  - The most effective ways to configure these messages depends heavily on what LLM is being used. Some instruct LLMs are trained to pay much more or much less attention to system messages. Some instruct LLMs have entirely different format expectations, as we will notice later.
  - In this example, the system message contains the few-shot "training set" of examples and the chat message contains the last topic that we want to generate a subject for.
- `temperature` - controls how much of the probability distribution the model will use when it is generating each token.
  - 1.0 means that it samples from the complete probability distribution
  - 0.7 means that it drops the bottom 30% of the least likely tokens when it is sampling
  - 0.0 means that it will perform deterministically and always output the single most probable token for each context
- `top_p` - is an alternative way of controling the sampling.
- `frequency_penalty` and `presence_penalty` - these are two ways of reduing the model from repeating the same words in one output. You can set these to be >0 if you're seeing a lot of repetition in your output.
- `max_tokens` - is the maximum length in tokens that will be output by calling the function. A token is a subword unit. There are roughly 2 or 3 tokens per word on average.
- `stop` - is a list of stop sequences. The model will stop generating output once it generates one of these strings, even if it hasn't reached the max token length.
  - By default this is set to some special token like `<|endoftext|>`.

You can read more about ways to use the Chat Completion API call in the [OpenAI Cookbook/example guide](https://cookbook.openai.com/examples/how_to_format_inputs_to_chatgpt_models) (note that this makes strong assumptions based on how GPT models from OpenAI are trained).
You can also read more about ways to design system and user messages to do specific tasks in the [Llama prompting guide](https://developer.meta.com/ai/docs/how-to-guides/prompting/) (note that these examples also make strong assumptions based on how their models are trained).

# Zero-shot learning

In addition to few shot learning, LLMs can sometimes also perform "zero shot learning" where instead of giving it several examples of what we want it to do, we can instead give it instructions of what we want it to do.

For example, for our topic-to-subject task we could give Llama-3.1-8B-Instruct a system message containing something like:

> Given a topic the user wants to learn about, output what subject they should study.

Then, we could append a user message:

> cellular respiration -

The LLM will output `biology` ... or something similar. Since there is no training and no example, you might have to accept slight variability in the output. `biology` and `biochemistry` are both reasonable outputs for the input `cellular respiration -`.

Implement a `generate_subject_zero_shot` function below that uses a zero-shot prompt.

In [12]:
def _chat(system_instruction, user_content):
    client = openai.OpenAI(
        base_url="https://integrate.api.nvidia.com/v1",
        api_key=os.environ['NVIDIA_API_KEY'],
    )
    response = client.chat.completions.create(
        model="meta/llama-3.1-8b-instruct",
        messages=[
            {"role": "system", "content": system_instruction},
            {"role": "user", "content": user_content},
        ],
        temperature=0.7, max_tokens=512, top_p=1,
    )
    time.sleep(1)
    return response.choices[0].message.content.strip()

def generate_subject_zero_shot(topic):
    return _chat(
        "Given a topic the user wants to learn about, output only the single "
        "school subject they should study. Respond with just the subject name.",
        topic + " - ",
    )
    pass

topic = "cellular respiration"
generate_subject_zero_shot(topic)

'Biology'

One very cool finding is that training procedure for LLMs can be changed to improve this instruction-following behavior. If LLMs are [trained to do multiple tasks through prompting](https://arxiv.org/abs/2110.08207), they better generalize to complete new tasks in a zero-shot fashion. In fact, the [prompt instructions may actually be more impactful than training data](https://arxiv.org/abs/2305.08298).

Try writing zero-shot prompts to do the following tasks:
1. Summarize a Wikipedia paragraph or short article.
2. Answer questions about a short article.
3. Re-write a short article so that it's suitable for a young child who is just learning how to read (age 8 or so).
4. Translate a sentence to English.

You should experiment with a few prompts in the NVIDIA playground (allows user message editing) and/or API (allows both system and user message editing) to find a good prompt combination that seems to work well.

In [13]:
def summarize(article_text):
    # TODO - write this function
    summary = _chat(
        "Summarize the following text in 2-3 sentences. Output only the summary.",
        article_text,
    )
    return summary


def answer_question(article_text, question):
    # TODO - write this function
    answer = _chat(
        "Answer the question using only the article below. Be concise.",
        f"Article:\n{article_text}\n\nQuestion: {question}",
    )
    return answer


def simplify(article_text):
    # TODO - write a function to re-write an article so that it's suitable for a young child.
    simplified_article = _chat(
        "Rewrite the text so an 8-year-old just learning to read can understand it. "
        "Use short, simple sentences.",
        article_text,
    )
    return simplified_article


def translate(sentence_text, source_language, target_language):
    # TODO - write a function to translate a sentence from a source language to a target language.
    # Note that this may only be feasible for common languages, because
    # you can't translate to/from a language that a model has never "seen"
    translated_sentence = _chat(
        f"Translate the following text from {source_language} to {target_language}. "
        "Output only the translation.",
        sentence_text,
    )
    return translated_sentence

What are some example inputs and matching outputs you get when running these functions?

The notebook file that you turn in should have these outputs for the course staff to review.

In [14]:
article_text = """
A triathlon is an endurance multisport race consisting of swimming, cycling, and running over various distances.[1] Triathletes compete for fastest overall completion time, racing each segment sequentially with the time transitioning between the disciplines included.[1][2][3] The word is of Greek origin, from τρεῖς (treîs), 'three', and ἆθλος (âthlos), 'competition'.[4]

The modern version of the sport originated in the late 1970s[5] in Southern California as sports clubs and individuals developed the sport. This history has meant that variations of the sport were created and still exist. It also led to other three-stage races using the name triathlon despite not being continuous or not consisting of swim, bike, and run elements.[6]

Triathletes train to achieve endurance, strength, and speed. The sport requires focused persistent and periodised training for each of the three disciplines, as well as combination workouts and general strength conditioning.[7]
"""

summarize(article_text)

'A triathlon is a multisport endurance race consisting of swimming, cycling, and running, where athletes compete for the fastest overall completion time. The sport originated in the late 1970s in Southern California and has since evolved into various forms. Triathletes require focused and periodized training to achieve endurance, strength, and speed across the three disciplines.'

In [15]:
article_text = """
A triathlon is an endurance multisport race consisting of swimming, cycling, and running over various distances.[1] Triathletes compete for fastest overall completion time, racing each segment sequentially with the time transitioning between the disciplines included.[1][2][3] The word is of Greek origin, from τρεῖς (treîs), 'three', and ἆθλος (âthlos), 'competition'.[4]

The modern version of the sport originated in the late 1970s[5] in Southern California as sports clubs and individuals developed the sport. This history has meant that variations of the sport were created and still exist. It also led to other three-stage races using the name triathlon despite not being continuous or not consisting of swim, bike, and run elements.[6]

Triathletes train to achieve endurance, strength, and speed. The sport requires focused persistent and periodised training for each of the three disciplines, as well as combination workouts and general strength conditioning.[7]
"""

questions = [
    "When were triathlons invented?",
    "What is a triathlon?",
    "What are the different parts of a triathlon?",
    "What does the word triathlon mean?",
    "What do triathletes train to achieve?",
]

for question in questions:
    answer = answer_question(article_text, question)
    print(question)
    print(answer)
    print('---')

When were triathlons invented?
The modern version of triathlons originated in the late 1970s in Southern California.
---
What is a triathlon?
A triathlon is an endurance multisport race consisting of swimming, cycling, and running over various distances.
---
What are the different parts of a triathlon?
The different parts of a triathlon are swimming, cycling, and running.
---
What does the word triathlon mean?
The word "triathlon" is of Greek origin, meaning 'competition of three'.
---
What do triathletes train to achieve?
Triathletes train to achieve endurance, strength, and speed.
---


In [16]:
article_text = """
In modern competition, triathletes often wear a watch to monitor time. This watch may be a smartwatch, tracking other metrics like distance, pace, and speed. A triathlete may also utilize a heart rate monitor, either on the wrist or across the chest, to help maintain a sustainable pace with maximum energy efficiency without "bonking" or "hitting the wall" (becoming too fatigued to continue).

"""

simplify(article_text)

'Here is the text rewritten for an 8-year-old:\n\nTriathletes wear a special watch. It helps them know how much time they spend doing sports. Some watches can even tell how far they run or swim. \n\nThey also wear a heart rate monitor. It\'s like a little machine that says, "Hey, you\'re working too hard! Slow down!" It helps the triathlete run or swim without getting too tired. That\'s called "bonking". When that happens, they can\'t keep going.'

In [17]:
spanish_sentence = """
En las competiciones modernas, los triatletas suelen llevar un reloj para controlar el tiempo.
"""

source_language = "Spanish"
target_language = "English"
translate(spanish_sentence, source_language, target_language)

'In modern competitions, triathletes usually wear a watch to control the time.'

## Pick your own task!

For this section you should pick some task that you'd like to have a LLM do. Add a description and code to your notebook here. You should:

1. Write a short description of what task you tried, and why you were interested in it.
2. Give some code so that we can reproduce what you did via an API call. You should include output of your code in the Python Notebook that you turned in.
3. Write a short qualitative analysis of whether or not Llama-3.1-8B-Instruct did the task well.

In [18]:
"""
I want to determine whether something is positive, negative, or neutral
"""

'\nI want to determine whether something is positive, negative, or neutral\n'

In [20]:
def classify_sentiment(text):
    return _chat(
        "Classify the sentiment of the text as positive, negative, or neutral. "
        "Respond with one word only.",
        text,
    )

In [21]:
classify_sentiment("The soccer game was fun but slightly rainy.")

'Neutral.'

In [ ]:
"""
TODO - write a short paragraph giving your qualitative analysis of how well the LLM did for your task.
"""

# Probabilities

Just like with the n-gram language models that we stuided earlier in the course, neural language models like Llama or GPT-4 assign probabilities to each token in a sequence.  

For example, if wrote a prompt like "Today I met Professor", the distribution over words to follow "Professor" might resemble something like this:
```
X = 27.80%
Jane = 23.01%
John = 7.89%
[ = 6.32%
L = 2.86%
Total: -2.54 logprob on 1 tokens
(67.88% probability covered in top 5 logits)
```
(note: if your model actually has this top-5 distribution, you may want to investigate your training dataset quality and especially figure out why those last two entries are there.)

## Using local LLMs

However, if we try to use the Nvidia NIM API to retrieve a probability distribution, we run into a problem. Nvidia allows free demo usage, but they (at least based on staff testing out the API) do not return any probability distribution information in their API. They also don't allow for any finetuning, which will be a problem in our next task section. (In general, finetuning requires more compute time and hardware than an equal number of inference calls - and if an API supports finetuning, then they need extra storage space to keep track of everyone's modified weights too. The estimated cost of finetuning GPT-3.5-turbo on the OpenAI API platform for our task and training dataset would be approximately $1 per attempt.)

Let's try to overcome this problem with our third model usage method: downloading and running a model locally.

For model size and runtime practicality reasons, we're going to test out our third model: [Gemma-2-2B-it](https://huggingface.co/google/gemma-2-2b-it).

This (inference) section of the homework assignment should be able to work, albeit slowly, if you run the notebook on **Google CoLab with a CPU** runtime environment.

The very last (finetuning) section of the homework assignment should be able to work if you run the notebook on **Google CoLab with a T4 GPU** runtime environment. (Some other options as of July 2026 include running the notebook on Amazon SageMaker Studio Lab, or running it on a departmental or personal machine with a CUDA-supporting GPU and at least 16GB GPU RAM.)

We will need to install the [HuggingFace Transformers library](https://huggingface.co/docs/transformers/installation) using `pip`. We will also install a few other packages so we can load, run, and finetune our local model more efficiently.

In [ ]:
%%capture
%pip install transformers peft accelerate bitsandbytes safetensors numpy trl wandb

Note: Interactive notebooks sometimes show strange behavior when you install new packages while the notebook is running. If you encounter issues, restart the notebook session and re-run all cells that you would have run *after* installing new packages.

If you are on CoLab, you can do this via "Runtime">"Restart session" in the top bar, after running the pip install commands above. This "resets" the Python interactive environment that the notebook uses.
- Do not "Disconnect and delete runtime" - this undoes the package installation and deletes any temporary saved files!
- Do not re-run the pip install cell, unless you were disconnected from the runtime machine!

In order to gain access to the Gemma-2-2B-it model weights on HuggingFace, you need to log in to your account and request access on the [model page](https://huggingface.co/google/gemma-2-2b-it).

You will also need to [create and retrieve your HuggingFace user access token](https://huggingface.co/settings/tokens) to indicate that you have permissions to download the model weights. We suggest you create a token with Read+Write permissions, as we will be finetuning and uploading a model later.

In [28]:
from getpass import getpass
import os

print('Enter HuggingFace access token:')
os.environ['HF_TOKEN'] = getpass().strip()
# you can create a key by going to https://huggingface.co/settings/tokens
# the HF_TOKEN environment variable is automatically used
# by the transformers library as appropriate

Enter HuggingFace access token:
··········


Once we have configured the Transformers library, let's load the Gemma model and do some of the same tasks that we did earlier: prompt completion, chat-formatted few-shot learning, and chat-formatted zero-shot learning.

We could try using the example code from [the model page](https://huggingface.co/google/gemma-2-2b-it). This type of approach is really straightforward and easy to read.

Note that the Gemma instruct models, unlike the Llama instruct models, do *not* use a "system" message. Instead, they are trained to expect alternating "user" and "assistant" messages.

```python
import torch
from transformers import pipeline

pipe = pipeline(
    "text-generation",
    model="google/gemma-2-2b-it",
    model_kwargs={"torch_dtype": torch.bfloat16},
    device="cuda",
)

messages = [
    {"role": "user", "content": "Who are you? Please, answer in pirate-speak."},
]

outputs = pipe(messages, max_new_tokens=256)
assistant_response = outputs[0]["generated_text"][-1]["content"].strip()
print(assistant_response)

# expecting output:
# Ahoy, matey! I be Gemma, a digital scallywag, a language-slingin' parrot of the digital seas. I be here to help ye with yer wordy woes, answer yer questions, and spin ye yarns of the digital world.  So, what be yer pleasure, eh? 🦜
```

Or, we might want a more detailed example to see what is going on inside of a pipeline. This type of approach gives us much more precise control over what's going on.

```python
from transformers import AutoTokenizer, AutoModelForCausalLM
import torch

tokenizer = AutoTokenizer.from_pretrained("google/gemma-2-2b-it")
model = AutoModelForCausalLM.from_pretrained(
    "google/gemma-2-2b-it",
    device_map="auto",
    torch_dtype=torch.bfloat16,
)

messages = [
    {"role": "user", "content": "Write me a poem about Machine Learning."},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    return_dict=True,
).to("cuda")

outputs = model.generate(**input_ids, max_new_tokens=256)
print(tokenizer.decode(outputs[0]))
```

However, unless you have a nice machine of your own, both of these configurations might be a little large for Google CoLab CPU inference and for finetuning later. We will [quantize](https://huggingface.co/docs/optimum/concept_guides/quantization) the models to reduce how much memory this model uses.

Note that the inference output includes our prompt. In fact, it includes a reformatted version of the prompt. The system and user message history are actually translated to text that are wrapped in special instruction tokens for this type of LLM task.

In [22]:
# Identify whether we are using CPU or GPU here
# If we have a GPU, this value should be True
USE_CUDA = False

device = 'cpu'
if USE_CUDA:
    device = 'cuda'

In [23]:
# Clear out our GPU memory just in case
import gc
import traceback
import sys
import torch

if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
if 'trainer' in globals():
    del trainer
with torch.no_grad():
    torch.cuda.empty_cache()
gc.collect()

# if you are still observing out-of-memory issues, you may need to restart the
# jupyter notebook kernel and re-run the specific subsection of the notebook
# that failed (ideally, after fixing whatever originally caused OOM problems if
# possible)

KeyboardInterrupt: 

In [26]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch

# This is the model we want to use
model_name = "google/gemma-2-2b-it"

# Downloading the model may take a while,
# depending on HuggingFace and CoLab server load.
# Course staff timed this as taking around 5min for the first run on CPU.
# Your experience may vary.

# This extra configuration helps us "quantize" the model - that is, shrink the
# model so that it can be represented and used with smaller-sized values.
# This slightly hurts model accuracy, since we're trading precision in exchange
# for better size and efficiency.
# Specifically, we are using 4-bit quantization.
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device,
    trust_remote_code=True,
)

ModuleNotFoundError: Could not import module 'AutoTokenizer'. Are this object's requirements defined correctly?

In [25]:
# Let's use the model and tokenizer we loaded earlier...

# Inference on CPU may take a while
# Course staff timed this as taking around 5min on CPU-only
# Your experience may vary.

# Now we write our input prompt!
messages = [
    {"role": "user", "content": "Write me a 4-line poem about Machine Learning."},
]
# Convert our input into token IDs so the model can process it
input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

# Do inference!
# (note that we are asking for 64 new tokens at most)
outputs = model.generate(**input_ids, max_new_tokens=64)

# Convert our output (more token IDs) back into words
print("---")
print(tokenizer.decode(outputs[0]))
print("---")

# Note that the previous results actually include our prompt as well
# We can trim out the prompt too
print("===")
print(tokenizer.decode(outputs[0, input_ids["input_ids"].shape[1]:]))
print("===")

NameError: name 'tokenizer' is not defined

You could adapt the system and user messages as appropriate to do few-shot and zero-shot learning, similar to how we used the API.

You can also do regular prompt completion, similar to how we used the playground:

You can read more about how to use the library here: [https://huggingface.co/docs/transformers/llm_tutorial](https://huggingface.co/docs/transformers/llm_tutorial)

In [ ]:
# assumes we have the same model and tokenizer loaded from earlier!
# let's ignore the instruction capabilities and do some text completion

prompt = "Here are 7 colors: red, orange,"
input_ids = tokenizer(prompt, return_tensors="pt")

outputs = model.generate(
    **input_ids,
    max_new_tokens=64,
    do_sample=True,
)

print("---")
print(tokenizer.decode(outputs[0]))
print("---")

print("===")
print(tokenizer.decode(outputs[0, input_ids["input_ids"].shape[1]:]))
print("===")

And finally, we can get a probability distribution:

In [ ]:
import numpy as np

# assumes we have the same model and tokenizer loaded from earlier!

def print_next_token_prob(prompt, model, tokenizer, top_k=5):
    model_inputs = tokenizer([prompt], return_tensors="pt").to(model.device)

    generated_ids = model.generate(
        **model_inputs,
        max_new_tokens=1,
        num_beams=top_k,
        num_return_sequences=top_k,
        return_dict_in_generate=True,
        output_scores=True,
        pad_token_id=tokenizer.eos_token_id,
    )
    transition_scores = model.compute_transition_scores(
        generated_ids.sequences,
        generated_ids.scores,
        normalize_logits=True,
    ).to('cpu')

    input_length = model_inputs.input_ids.shape[1]
    generated_tokens = generated_ids.sequences[:, input_length:]
    print('===')
    for i in range(transition_scores.shape[0]):
        for tok, score in zip(generated_tokens[i], transition_scores[i]):
            # | token | token string | logits | probability
            print(f"| {tok:5d} | {tokenizer.decode(tok):8s} | {score.numpy():.4f} | {np.exp(score.numpy()):.2%}")
    print('===')

print_next_token_prob("Apples are very", model, tokenizer, top_k=5)

Let's return to why we care about probability distributions.

One critical observation about language models is that they often encode societal biases that appear in their data. For instance, after the discovery that LM embeddings could be used to solve word analogy problems like "**man** is to **woman** as **king** is to ___" (the model predicts **queen**), researchers discovered that LMs had a surpisingly sexist answer to the analogy problem  "**man** is to **woman** as **computer programmer** is to ___" (the model predicts **homemaker**). These kinds of biases are prevalent and pernicious.

Let's examine the most probable names that our LLM assigns to different completions and try to infer their gender. We'll see if it associates different genders with different academic disciplines. (You can also see this for different careers like *nurse*, *plumber*, or *school teacher*).

Please create dictionaries mapping the LLM's predictions for the first names of professors in these departments:
* Computer Science
* Gender Studies
* Physics
* Linguistics
* Bioengineering

Use the following prompt for each of the above departments and note the distribution of names & genders that GPT generates for each professor:
> My favorite class in the {department_name} Department was taught by Professor

Note: You can adjust `top_k` to change the number of candidate next tokens.

In [ ]:
# Put whatever code you ran to obtain these most-probable lists of names in here
# TODO
for dept in ["Computer Science", "Gender Studies", "Physics", "Linguistics", "Bioengineering"]:
    print(dept)
    print_next_token_prob(
        f"My favorite class in the {dept} Department was taught by Professor",
        model, tokenizer, top_k=5,
    )
    print()

In [ ]:
# Get the most likely name predictions
# Label each name as probably male, female, partial word, or unsure/unknown
computer_science_genders = {
    "TODO1" : "TODO",
    "TODO2" : "TODO",
    "TODO3" : "TODO",
    "TODO4" : "TODO",
    "TODO5" : "TODO",
}

gender_studies_genders = {
    "TODO1" : "TODO",
    "TODO2" : "TODO",
    "TODO3" : "TODO",
    "TODO4" : "TODO",
    "TODO5" : "TODO",
}

physics_genders = {
    "TODO1" : "TODO",
    "TODO2" : "TODO",
    "TODO3" : "TODO",
    "TODO4" : "TODO",
    "TODO5" : "TODO",
}

lingusitics_genders = {
    "TODO1" : "TODO",
    "TODO2" : "TODO",
    "TODO3" : "TODO",
    "TODO4" : "TODO",
    "TODO5" : "TODO",
}

bioengineering_genders = {
    "TODO1" : "TODO",
    "TODO2" : "TODO",
    "TODO3" : "TODO",
    "TODO4" : "TODO",
    "TODO5" : "TODO",
}

The sample size is low, but you can take a moment to reflect: does the model seem to obviously associate different disciplines with professors of a specific gender?

In [ ]:
"""
[Optional: Leave some notes about anything particularly interesting you learned from this section here]
"""

# Fine-Tuning

In addition to zero-shot and few-shot learning, another way of getting large language models to do your tasks is via a process called "fine tuning". In fine-tuning, the model updates its parameters so that it performs well on many training examples. The training examples are in the form of input prompts paired with gold standard completions.

Large language models are pre-trained to perform well on general tasks like text completion, but not necessarily on the specific task that you might be interested in. The models can be finetuned to perform your task, starting with the model parameters that are good for the general setting, and then updating them to be better at your task.

We'll walk through how to fine-tune Gemma for a task, using the Transformers library.

**Note that this section will require a GPU runtime environment, so we have included copies of package installation, API key entry, and model-loading code to rerun as appropriate.**

In [ ]:
%%capture
%pip install transformers torch peft accelerate bitsandbytes safetensors numpy trl wandb

For this example task, we will show you how to fine-tune our LLM to write biographies when given data in the info boxes in Wikipedia pages.  For instance, given this input

```
notable_type: scientist
name: Zulima Aban
gender: female
birth_date: 05 December 1905
birth_place: Valencia, Spain
death_date: 09 August 1983
death_place: Detroit, Michigan, U.S.
death_cause: Pulmonary embolism
occupation: Astronomer
fields: Astrophysics, Computer Science, Computer Graphics, Interface Design, Image Synthesis
known_for: The Search for Planet Nine
hometown: Detroit, Michigan, U.S.
nationality: Venezuelan
citizenship: Spanish, American
alma_mater: University of Valencia (B.Sc.), University of Madrid (Ph.D.)
thesis_title: The Formation of Planets by the Accretion of Small Particles
thesis_year: 1956
doctoral_advisor: Angela Carter
awards: Spanish Academy of Science, Spanish Academy of Engineering, German Aerospace Prize, IEEE Medal of Honor, IEEE John von Neumann Medal, IEEE Jack S. Kilby Signal Processing Medal, United Nations Space Pioneer Award, Wolf Prize in Physics
institutions: Oberlin College, University of Valencia, Instituto de Astrofísica de Andalucía (CSIC), University of Southern California, Space Telescope Science Institute (STScI)
notable_students: Ryan Walls
influences: Immanuel Kant, Albert Einstein, Kurt Gödel, Gottfried Leibniz, Richard Feynman, Werner Heisenberg, William Kingdon Clifford, Sir Arthur Eddington
influenced: Joseph Weinberg
mother: Ana Aban
father: Joaquín Aban
partner: Georgina Abbott
children: Robert, Peter, Sarah
```

The fine-tuned model should generate something like this output:

> Zulima Aban was a Venezuelan astronomer, who was born on 05 December 1905 in Valencia, Spain to Ana Aban and Joaquín Aban. Her career involved the fields of Astrophysics, Computer Science, Computer Graphics, Interface Design, Image Synthesis. Aban was known for The Search for Planet Nine. Aban went to University of Valencia (B.Sc.), University of Madrid (Ph.D.). Aban's thesis title was The Formation of Planets by the Accretion of Small Particles in 1956. Her doctoral advisor was Angela Carter. Aban received Spanish Academy of Science, Spanish Academy of Engineering, German Aerospace Prize, IEEE Medal of Honor, IEEE John von Neumann Medal, IEEE Jack S. Kilby Signal Processing Medal, United Nations Space Pioneer Award, Wolf Prize in Physics. Aban went to Oberlin College, University of Valencia, Instituto de Astrofísica de Andalucía (CSIC), University of Southern California, Space Telescope Science Institute (STScI). Her notable students were Ryan Walls. Aban was influenced by Immanuel Kant, Albert Einstein, Kurt Gödel, Gottfried Leibniz, Richard Feynman, Werner Heisenberg, William Kingdon Clifford, Sir Arthur Eddington and she infuenced Joseph Weinberg. Aban was married to Georgina Abbott and together had three children, Robert, Peter, Sarah. Aban died on 09 August 1983 in Detroit, Michigan, U.S due to Pulmonary embolism.

The dataset that we will use was created for the paper [SynthBio: A Case Study in Human-AI Collaborative Curation of Text Datasets](https://www.cis.upenn.edu/~ccb/publications/synthbio.pdf) by Ann Yuan, Daphne Ippolito, Vitaly Nikolaev, Chris Callison-Burch, Andy Coenen, and Sebastian Gehrmann. It was published in NeurIPS 2021. The goal of the paper was to create a curated dataset for training large language models on synthetic data with the goal of avoiding the gender and geographic bias that is naturally present in Wikipedia due to cultural and historic reasons.

## Load the data

In [ ]:
!wget https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_train.json

# If you're running this locally on a Mac, you can use the following command instead.
# !curl -o SynthBio_train.json https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_train.json

In [ ]:
# Load a file called 'SynthBio_train.json' which is a list of json objects.

import json
import random

def load_wiki_bio_data(filename='SynthBio_train.json', num_bios=100, randomized=True):
    with open(filename) as f:
        synth_bio_data = json.load(f)
    random.shuffle(synth_bio_data)
    bios = []
    for data in synth_bio_data:
        notable_type = data['notable_type']
        attributes = "notable_type: {notable_type} | {other_attributes}".format(
            notable_type = notable_type,
            other_attributes = data['serialized_attrs']
        )
        biography = data['biographies'][0]
        bios.append((attributes.replace(" | ", "\n"), biography))
    return bios[:min(num_bios, len(bios))]

wiki_bios = load_wiki_bio_data()
attributes, bio = wiki_bios[0]
print(attributes)
print('---')
bio

## Format data for fine-tuning

Below, we show how to format data to fine-tune a model using the Transformers library. We will be using the [SFTTrainer](https://huggingface.co/docs/trl/en/sft_trainer) that HuggingFace provides. The basic format needed for our purposes is some collection of texts which match the system, user, and assistant message format that we would like to use later. For example, our finetuning training dataset for Gemma should look something like this (because Gemma does not use system messages):

```
{"messages": [{"role": "user", "content": "User's prompt 1"}, {"role": "assistant", "content": "Expected response 1."}]}
{"messages": [{"role": "user", "content": "User's prompt 2"}, {"role": "assistant", "content": "Expected response 2."}]}
{"messages": [{"role": "user", "content": "User's prompt 3"}, {"role": "assistant", "content": "Expected response 3."}]}
...
```

In the code below, we'll extract a prompt that contains the `attributes` variable from the intent determination data, and we'll have the completion be the `biography` variable.

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

def create_wikibio_generator_finetuning_data(wikibios):
    all_data = []
    general_instructions =  "\n".join([
        "Generate a biography for a person based on the details provided.",
        "Output only a single paragraph.",
        "Do not include extra questions.",
    ])

    for attributes, bio in wiki_bios:
        all_data.append([
            {
                "role": "user",
                "content": "{general_instructions}\n\n{attributes}\n".format(
                    general_instructions=general_instructions,
                    attributes=attributes,
                ),
            },
            {"role": "assistant", "content": bio,}
        ])

    random.shuffle(all_data)
    return Dataset.from_pandas(pd.DataFrame({
        "messages": all_data
    }))

# Format our training data source file
wiki_bios = load_wiki_bio_data()
finetuning_dataset = create_wikibio_generator_finetuning_data(wiki_bios)

# Split our training dataset into 90% training / 10% held-out
finetuning_dataset_split = finetuning_dataset.train_test_split(test_size=0.1)
# We could split this further into train/eval/test if we needed
# But we actually don't need to, in this case
# (there is actually a separate test dataset file that we use later)
print(f"  Training n={len(finetuning_dataset_split['train'])}")
print(f"Validation n={len(finetuning_dataset_split['test'])}")

# put these all back into a single DatasetDict
finetuning_dataset = DatasetDict({
    "train": finetuning_dataset_split['train'],
    "eval": finetuning_dataset_split['test'],
})
finetuning_dataset.save_to_disk("wikibio_boxtobio_finetuning_data")

We have formatted our finetuning data as an instruction task, which still has a regular instruction prompt. Our goal here is to "clarify" the style that the instruct model uses, instead of "teaching" new fundamental knowledge.

This saves our dataset in a nice convenient format that we can reload later...

Note: The question of when finetuning a base model and then re-training it to follow instructions, vs. finetuning an instruct model is better, or how best to efficiently update an instruction model in general, is a topic of active research. Here's [one example paper](https://huggingface.co/papers/2505.12716) that examines this question and suggests a better approach. Unfortunately, while their method may be effective, it's also not widely adopted enough yet and therefore infeasible for us to attempt.

In [ ]:
from datasets import DatasetDict

# this is loaded as a DatasetDict because it contains both train and eval
finetuning_dataset = DatasetDict.load_from_disk("wikibio_boxtobio_finetuning_data")
finetuning_dataset

In [ ]:
finetuning_dataset['train'][0]

In [ ]:
finetuning_dataset['eval'][0]

Next, we'll perform fine-tuning with this data.

## Run fine-tuning with SFTTrainer

Let's finetune our model. This isn't necessarily the *best* finetuning we could do, but it will teach our model at least a little bit.

By default, we have WandB training logging enabled, so this also needs a WandB API key.

Credits for assembling this training pipeline go to several sources:
- General HuggingFace guide: https://huggingface.co/docs/transformers/training
  - https://github.com/huggingface/notebooks/blob/main/examples/language_modeling.ipynb
- Original Gemma cookbook guide: https://huggingface.co/blog/gemma-peft
- Instruct model finetuning: https://towardsdatascience.com/fine-tune-an-instruct-model-over-raw-text-data-6db654e7e2ed/
  - https://colab.research.google.com/drive/11X5ptOe3zbFE2s1AeHu-gynwAbkE-7Zn#scrollTo=yg3aXa0mYp6Q
- PEFT adapter saving and loading: https://discuss.huggingface.co/t/correct-way-to-save-load-adapters-and-checkpoints-in-peft/77836/2
  - https://colab.research.google.com/drive/12pMorxvLV-VwjuNBM76L4xXnzVYg57iB?usp=sharing#scrollTo=Gzke5ccani9m

In [ ]:
from getpass import getpass
import os

print('Enter HuggingFace access token: hf_REDACTED')
os.environ['HF_TOKEN'] = getpass().strip()
# you can create a key by going to https://huggingface.co/settings/tokens
# the HF_TOKEN environment variable is automatically used
# by the transformers library as appropriate

In [27]:
from getpass import getpass
import os

print('Enter a Weights and Biases API key:')
os.environ['WANDB_API_KEY'] = getpass().strip()
# you can create a key by going to https://wandb.ai/authorize?ref=models
# the WANDB_API_KEY environment variable is automatically used
# by the wandb library as appropriate

Enter a Weights and Biases API key: REDACTED
··········


In [ ]:
# Identify whether we are using CPU or GPU here
# If we have a GPU, this value should be True
USE_CUDA = True

device = 'cpu'
if USE_CUDA:
    device = 'cuda'

In [ ]:
# Clear out our GPU memory just in case
import gc
import traceback
import sys
import torch

if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
if 'trainer' in globals():
    del trainer
with torch.no_grad():
    torch.cuda.empty_cache()
gc.collect()

# if you are still observing out-of-memory issues, you may need to restart the
# jupyter notebook kernel and re-run the specific subsection of the notebook
# that failed (ideally, after fixing whatever originally caused OOM problems if
# possible)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch
from peft import (
    LoraConfig,
    get_peft_model
)

# Which model are we training?
model_name = "google/gemma-2-2b-it"

# Let's built the name that we want to save our new model as
my_hf_username = "username" # TODO: UPDATE THIS TO MATCH YOUR HF USERNAME
new_model_name = "cs5210-26su-hw10-finetuned-boxtobio"

# This is where our model would be saved on HuggingFace
hub_model_id = f"{my_hf_username}/{new_model_name}"

# This is what project our work gets noted under on WandB
# You can see your projects at https://wandb.ai/[username]/projects
os.environ['WANDB_PROJECT'] = "cs5210-26su-hw10"

# We quantize our model again
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name,
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map=device,
    trust_remote_code=True,
)

# Updating all 2B of the parameters would be a lot of work, but we can do more
# efficient learning by focusing only on the most relevant parameters!
# Here, we specify which exact bits in the model we're allowing training for
# You can find more information about what each of these arguments means at:
# https://huggingface.co/docs/peft/main/en/package_reference/lora#peft.LoraConfig
lora_config = LoraConfig(
    r=8,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)

# Print some numbers about how many weights we are training
model.print_trainable_parameters()

Before we actually start finetuning, let's get a quick example of what our original model behavior would be on this task.

In [ ]:
messages = [
    {"role": "user", "content": 'Generate a biography for a person based on the details provided.\nOutput only a single paragraph.\nDo not include extra questions.\n\nnotable_type: musician\nname: Thomas Djambo\nbirth_name: Jean Thomas Djambo\nalias: Djambo\ngender: male\nbirth_date: 15 October 1901\nbirth_place: Lambarene, Gabon\ndeath_date: 1976\ndeath_cause: Heart Attack\nresting_place: Lambarene, Gabon\ninstrument: Saxophone\ngenre: Jazz\nhometown: Lambarene, Gabon\nnationality: Gabonese\ncitizenship: Gabon\neducation: Self-taught musical prodigy\nyears_active: 1920-1976\nlabel: none\nmother: unknown\nfather: unknown\npartner: Jean Afrique\n'},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

outputs = model.generate(**input_ids, max_new_tokens=256)

print(tokenizer.decode(outputs[0]))

Note that its original behavior is quite decent already, since our prompt included clear instructions. Our goal is to refine the model slightly to better suit our needs.

Let's start training!

In [ ]:
import transformers
from trl import SFTTrainer
import wandb

# Course staff timed this as taking around 15min on a T4 GPU
# Your experience may vary.

# You can find more information about each of these arguments at:
# https://huggingface.co/docs/trl/main/en/sft_trainer#trl.SFTTrainer
# https://huggingface.co/docs/trl/main/en/sft_trainer#trl.SFTConfig
# (check the appropriate package version as necessary)
trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=finetuning_dataset['train'],
    eval_dataset=finetuning_dataset['eval'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2,
        per_device_eval_batch_size=4,
        # gradient_accumulation_steps=4,
        warmup_steps=2,
        num_train_epochs=2,
        optim="paged_adamw_8bit",
        learning_rate=2e-5,
        weight_decay=0.001,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        eval_strategy="steps",
        logging_steps=0.1,
        save_strategy="best",
        load_best_model_at_end=True,
        output_dir="outputs",
        report_to="wandb",
        hub_model_id = hub_model_id,
        # push_to_hub=True, # save to the hub every time we'd locally save?
        # hub_private_repo=True,
    ),
)
trainer.train()

# How well does it do on our test set?
# ONLY CALL THIS WHEN WE ARE SURE WE WANT TO CHECK
# in this case, we didn't actually make a dedicated test dataset
# results = trainer.evaluate(finetuning_dataset['test'])
# print(results)

# Force the WandB tracking session to end
wandb.finish()


Let's save our finetuned model to the HuggingFace model hub.

(This code saves your model publicly by default.)

In [ ]:
# manually create the results repository
# by default, this will be public!
trainer.push_to_hub(hub_model_id)

## Use your fine-tuned model

We can see the training loss and mean token accuracy improved over time as it trained. Let's actually use this model for biography generation now.

If you have successfully pushed the finetuned model to the HuggingFace model hub (this requires a HF access token with some kind of write access), you can also load the finetuned model in directly later, similar to how we loaded the base model weights in directly.

In [ ]:
# Clear out our GPU memory just in case
import gc
import traceback
import sys
import torch

if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
if 'trainer' in globals():
    del trainer
with torch.no_grad():
    torch.cuda.empty_cache()
gc.collect()

# if you are still observing out-of-memory issues, you may need to restart the
# jupyter notebook kernel and re-run the specific subsection of the notebook
# that failed (ideally, after fixing whatever originally caused OOM problems if
# possible)

In [ ]:
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    BitsAndBytesConfig,
)
import torch
from peft import (
    LoraConfig,
    get_peft_model
)

# Quantize for inference
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

# Load the model and tokenizer
tokenizer = AutoTokenizer.from_pretrained(
    model_name, # our model uses the same tokenizer as the base one
    trust_remote_code=True,
)
model = AutoModelForCausalLM.from_pretrained(
    hub_model_id, # we are loading OUR model
    quantization_config=bnb_config,
    device_map=device,
    trust_remote_code=True,
)

# if we needed to continue training, we can do this:
# underneath it all, our "new model" contains only the weights that we adjusted
# so loading a model means retrieving those weights, and then combining them
# with the base model weights
# model = PeftModel.from_pretrained(
#     model_name, # base model ID
#     hub_model_id, # our finetuned model ID
#     is_trainable=True,
# )
# model.print_trainable_parameters()

In [ ]:
# We can use it for inference directly
messages = [
    {"role": "user", "content": 'Generate a biography for a person based on the details provided.\nOutput only a single paragraph.\nDo not include extra questions.\n\nnotable_type: musician\nname: Thomas Djambo\nbirth_name: Jean Thomas Djambo\nalias: Djambo\ngender: male\nbirth_date: 15 October 1901\nbirth_place: Lambarene, Gabon\ndeath_date: 1976\ndeath_cause: Heart Attack\nresting_place: Lambarene, Gabon\ninstrument: Saxophone\ngenre: Jazz\nhometown: Lambarene, Gabon\nnationality: Gabonese\ncitizenship: Gabon\neducation: Self-taught musical prodigy\nyears_active: 1920-1976\nlabel: none\nmother: unknown\nfather: unknown\npartner: Jean Afrique\n'},
]
input_ids = tokenizer.apply_chat_template(
    messages,
    return_tensors="pt",
    return_dict=True,
).to(model.device)

outputs = model.generate(**input_ids, max_new_tokens=64)

print(tokenizer.decode(outputs[0]))

In [ ]:
# We can also call a function...
def generate_bio(attributes, model, tokenizer):
    # Set up the prompt
    general_instructions =  "\n".join([
        "Generate a biography for a person based on the details provided.",
        "Output only a single paragraph.",
        "Do not include extra questions.",
    ])
    messages = [
        {
            "role": "user",
            "content": "{general_instructions}\n\n{attributes}\n".format(
                general_instructions=general_instructions,
                attributes=attributes,
            ),
        },
    ]
    # Generate results
    input_ids = tokenizer.apply_chat_template(
        messages,
        return_tensors="pt",
        return_dict=True,
    ).to(model.device)
    output_ids = model.generate(
        **input_ids,
        max_new_tokens=256,
        top_p=1,
        do_sample=True,
    )
    # Get some results that are only the body of text following the prompt
    return tokenizer.decode(
        output_ids[0, input_ids["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

In [ ]:
attributes = """notable_type: computer scienist
alma_mater: Stanford University (BS in Symbolic Systems), University of Edinburgh (PhD in Informatics)
birth_place: California
children: 2
gender: male
main_interests: Artificial Intelligence, Natural Language Processing
name: Chris Callison-Burch
nationality: American
notable_works: Moses: Open source toolkit for statistical machine translation, The Paraphrase Database (PPDB)
occupation: professor
courses_taught: AI, Crowdsourcing and NLP
enrollment_in_most_popular_course: 570 students
institution: University of Pennsylvania"""

biography = generate_bio(attributes, model, tokenizer)
print('---')
print(attributes)
print('---')
biography

## Analyze your model's output

Sometimes the model will add facts that are not present in the attributes.  For instance, one time it said

> He was a member of the research staff at IBM Research in Yorktown Heights.

which is not correct. Another time it said

> His most popular course was on AI, which had 570 students.

which is correct, but not specified in the attributes. (the attributes only say that some course had 570 students, not necessarily AI)

Try running your own fine-tuned model until it produces something that wasn't licensed by the attributes.

Save at least two good runs and one bad run below.

In [ ]:
generations_with_correct_facts = [
    """ TODO 1: a complete generation containing correct facts only """,
    """ TODO 2: a complete generation containing correct facts only """,
]

generation_with_incorrect_facts = """
TODO 1: a complete generation containing at least some incorrect facts
"""

incorrect_facts = [
    """ TODO: a list containing each sentence from
    generation_with_incorrect_facts that contains incorrect facts """,
]

## Fine-tune a new model!

Now that you've seen an example of how to fine-tune a local model with the SFTTrainer, let's have you write code to fine-tune your own model.

For this model, I'd like you to do the reverse direction of what we just did. Given a (noticeably out-of-date) Wikipedia Biograph like this:

> Jill Tracy Jacobs Biden (born June 3, 1951) is an American educator and the current first lady of the United States as the wife of President Joe Biden. She was the second lady of the United States from 2009 to 2017. Since 2009, Biden has been a professor of English at Northern Virginia Community College.
>
> She has a bachelor's degree in English and a doctoral degree in education from the University of Delaware, as well as master's degrees in education and English from West Chester University and Villanova University. She taught English and reading in high schools for thirteen years and instructed adolescents with emotional disabilities at a psychiatric hospital. From 1993 to 2008, Biden was an English and writing instructor at Delaware Technical & Community College. Biden is thought to be the first wife of a vice president or president to hold a paying job during her husband's tenure.
>
> Born in Hammonton, New Jersey, she grew up in Willow Grove, Pennsylvania. She married Joe Biden in 1977, becoming stepmother to Beau and Hunter, his two sons from his first marriage. Biden and her husband also have a daughter together, Ashley Biden, born in 1981. She is the founder of the Biden Breast Health Initiative non-profit organization, co-founder of the Book Buddies program, co-founder of the Biden Foundation, is active in Delaware Boots on the Ground, and with Michelle Obama is co-founder of Joining Forces. She has published a memoir and two children's books.

Your model should try to output something like this:

```
notable_type: First Lady of the United States
name: Jill Biden
gender: female
nationality: American
birth_date: 03 June 1951
birth_place: Hammonton, New Jersey
alma_mater: University of Delaware
occupation: professor of English at Northern Virginia Community College
notable_works: children's books and memoir
main_interests: education, literacy, women's health
partner: Joe Biden
children: Ashley Biden, Beau Biden (stepson), Hunter Biden (stepson)
```

In [ ]:
import pandas as pd
from datasets import Dataset, DatasetDict

def create_wikibio_parser_finetuning_data(wikibios):
    # TODO - write your fine-tuning dataset-building function
    all_data = []
    instructions = "\n".join([
        "Extract the attributes from the biography below.",
        "Output each attribute on its own line as 'attribute: value'.",
    ])
    for attributes, bio in wikibios:
        all_data.append([
            {"role": "user", "content": f"{instructions}\n\n{bio}\n"},
            {"role": "assistant", "content": attributes},
        ])
    random.shuffle(all_data)
    return Dataset.from_pandas(pd.DataFrame({"messages": all_data}))

# Format our training data source file
wiki_bios = load_wiki_bio_data()
finetuning_dataset = create_wikibio_parser_finetuning_data(wiki_bios)

# Split our training dataset into 90% training / 10% held-out
# Feel free to adjust this as appropriate!
finetuning_dataset_split = finetuning_dataset.train_test_split(test_size=0.1)
print(f"  Training n={len(finetuning_dataset_split['train'])}")
print(f"Validation n={len(finetuning_dataset_split['test'])}")

# put these all back into a single DatasetDict
finetuning_dataset = DatasetDict({
    "train": finetuning_dataset_split['train'],
    "eval": finetuning_dataset_split['test'],
})
finetuning_dataset.save_to_disk("wikibio_biotobox_finetuning_data")

In [ ]:
from datasets import DatasetDict

# this is loaded as a DatasetDict because it contains both train and eval
finetuning_dataset = DatasetDict.load_from_disk("wikibio_biotobox_finetuning_data")
finetuning_dataset

In [ ]:
# Run this if you have any previous model and tokenizer loaded into memory

# This is especially important since we'd like to finetune a *new* model to do this task!
# We do not want to finetune our already-finetuned-earlier model!

# Clear out GPU memory
import gc
import traceback
import sys
import torch

if 'model' in globals():
    del model
if 'tokenizer' in globals():
    del tokenizer
if 'trainer' in globals():
    del trainer
with torch.no_grad():
    torch.cuda.empty_cache()
gc.collect()

# if you are still observing out-of-memory issues, you may need to restart the
# jupyter notebook kernel and re-run the specific subsection of the notebook
# that failed (ideally, after fixing whatever originally caused OOM problems if
# possible)

In [ ]:
# TODO: PUT ALL OF YOUR FINETUNING CODE HERE

import transformers
from trl import SFTTrainer
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model
import torch, wandb, os

model_name = "google/gemma-2-2b-it"
my_hf_username = "USERNAME"
new_model_name = "cs5210-26su-hw10-finetuned-biotobox"
hub_model_id = f"{my_hf_username}/{new_model_name}"
os.environ['WANDB_PROJECT'] = "cs5210-26su-hw10"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    model_name, quantization_config=bnb_config, device_map=device, trust_remote_code=True,
)
lora_config = LoraConfig(
    r=8,
    target_modules=["q_proj","k_proj","v_proj","o_proj","gate_proj","up_proj","down_proj"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=finetuning_dataset['train'],
    eval_dataset=finetuning_dataset['eval'],
    args=transformers.TrainingArguments(
        per_device_train_batch_size=2, per_device_eval_batch_size=4,
        warmup_steps=2, num_train_epochs=2, optim="paged_adamw_8bit",
        learning_rate=2e-5, weight_decay=0.001,
        fp16=not torch.cuda.is_bf16_supported(), bf16=torch.cuda.is_bf16_supported(),
        eval_strategy="steps", logging_steps=0.1,
        save_strategy="best", load_best_model_at_end=True,
        output_dir="outputs", report_to="wandb", hub_model_id=hub_model_id,
    ),
)
trainer.train()
wandb.finish()
trainer.push_to_hub(hub_model_id)

In [ ]:
def parse_bio(biography, model, tokenizer):
    # TODO given a biography using your fine-tuned model,
    # return a single string representing the attributes
    instructions = "\n".join([
        "Extract the attributes from the biography below.",
        "Output each attribute on its own line as 'attribute: value'.",
    ])
    messages = [{"role": "user", "content": f"{instructions}\n\n{biography}\n"}]
    input_ids = tokenizer.apply_chat_template(
        messages, return_tensors="pt", return_dict=True,
    ).to(model.device)
    output_ids = model.generate(**input_ids, max_new_tokens=256, do_sample=False)
    return tokenizer.decode(
        output_ids[0, input_ids["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()

In [ ]:
# TODO you may want to do a dedicated test call on parse_bio
test_bio = wiki_bios[0][1]  # a biography string from the loaded data
print(parse_bio(test_bio, model, tokenizer))

## Test your parser

Next we will test your parser.  This will involve calling your `parse_bio` function about 250 times, so be sure that you've got it properly debugged and working before running this code.

In [ ]:
!wget https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_test.json

# If you're running this locally on a Mac, you can use the following command instead.
# !curl -o SynthBio_test.json https://raw.githubusercontent.com/artificial-intelligence-class/artificial-intelligence-class.github.io/master/homeworks/large-LMs/SynthBio_test.json

In [ ]:
import json

def load_wiki_bio_test_set(filename='SynthBio_test.json', max_test_items=10, randomized=True):
    """
    Loads our wikibio test set, and returns a list of tuples
    biographies (text), attributes (dictionaries)
    """
    with open(filename) as f:
        synth_bio_data = json.load(f)
    bios = []
    for data in synth_bio_data:
        notable_type = data['notable_type']
        attributes = data['attrs']
        attributes['notable_type'] = notable_type
        biography = data['biographies'][0]
        bios.append((biography, attributes))
    return bios[:min(max_test_items, len(bios))]

def convert_to_dict(predicted_attributes_txt):
    """
    Converts predicted attributes from text format into a dictionary.
    """
    predicted_attributes = {}
    for line in predicted_attributes_txt.split('\n'):
        try:
            attribute, value = line.split(':')
            predicted_attributes[attribute.strip()] = value.strip()
        except ValueError:
            continue
    return predicted_attributes

Helper functions for computing precision, recall and f-score.

In [ ]:
from collections import Counter

def update_counts(gold_attributes, predicted_attributes, true_positives, false_positives, false_negatives, all_attributes):
    # Compute true positives and false negatives
    for attribute in gold_attributes:
        all_attributes[attribute] += 1
        if attribute in predicted_attributes:
            # gold attribute exists in predicted attributes
            # some attributes have multiple values
            predicted_values = predicted_attributes[attribute].split(',')
            gold_values = gold_attributes[attribute].split(',')
            for value in gold_values:
                if value.strip() in predicted_values:
                    # gold attribute exists in predicted attributes and gold value exists in predicted values
                    true_positives[attribute] += 1
                else:
                    # gold attribute exists in predicted attributes but gold value does not exist in predicted values
                    false_negatives[attribute] += 1
        else:
            # gold attribute does not exist in predicted attributes
            false_negatives[attribute] += 1

    # Compute false positives
    for attribute in predicted_attributes:
        if attribute not in gold_attributes:
            # predicted attribute does not exist in gold attributes
            all_attributes[attribute] += 1
            false_positives[attribute] += 1
        else:
            # predicted attribute exists in gold attributes
            # some attributes have multiple values
            gold_values = gold_attributes[attribute].split(',')
            predicted_values = predicted_attributes[attribute].split(',')
            for value in predicted_values:
                if value.strip() not in gold_values:
                    # predicted attribute exists in gold attributes but predicted value does not exist in gold values
                    false_positives[attribute] += 1

In [ ]:
def evaluate_on_test_set(model, tokenizer, wiki_bio_test, threshold_count = 5):
  """
  Computer the precision, recall and f-score for each of the attributes
  that appears more than the threshold count
  """
  true_positives = Counter()
  false_positives = Counter()
  false_negatives = Counter()
  all_attributes = Counter()

  for bio, gold_attributes in wiki_bio_test:
      predicted_attributes = convert_to_dict(parse_bio(bio, model, tokenizer))
      update_counts(gold_attributes, predicted_attributes, true_positives, false_positives, false_negatives, all_attributes)

  average_precision = 0
  average_recall = 0
  total = 0

  for attribute in all_attributes:
      if all_attributes[attribute] < threshold_count:
          continue
      print(attribute.upper())
      try:
          precision = true_positives[attribute] / (true_positives[attribute] + false_positives[attribute])
      except:
          precision = 0.0
      try:
          recall = true_positives[attribute] / (true_positives[attribute] + false_negatives[attribute])
      except:
          recall = 0.0
      print("precision:", precision)
      print("recall:", recall)
      print("f-score:", (precision+recall)/2)
      print('---')
      average_precision += precision
      average_recall += recall
      total += 1

  print("AVERAGE")
  average_precision = average_precision/total
  average_recall = average_recall/total
  print("precision:", average_precision)
  print("recall:", average_recall)
  print("f-score:", (average_precision+average_recall)/2)
  print('---')

If you would like to evaluate on the full test set, there are 237 test items.  You can set `max_test_items=237`.  Doing so will call your `parse_bio` function about 237 times (this would take approximately 120min based on our staff testing), so be sure that you've got it properly debugged and working before running this code.

### **Important: Grading Criteria**

For a thorough evaluation of the model, max_test_items has to be set to a **minimum of 50**. To receive full credit, you have to submit a previously run notebook with all outputs included. Full credit will be awarded if **at least two out of three metrics - average precision, average recall, and average f-score - exceed 0.10 (10%) on at least 50 test set examples**.

We can definitely do better than this using the same model with some higher-effort finetuning and higher-quality postprocessing, but 10% is a low-bounds threshold for our purposes.

In [ ]:
# Load your finetuned model

from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True, bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
)
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    hub_model_id, quantization_config=bnb_config, device_map=device, trust_remote_code=True,
)

In [ ]:
testset_filename = 'SynthBio_test.json'
max_test_items = 10  # set to 10 for initial testing; for final testing and submission, choose a value between 50 and 237
wiki_bio_test = load_wiki_bio_test_set(testset_filename, max_test_items)
evaluate_on_test_set(model, tokenizer, wiki_bio_test, threshold_count=5)

How well did your model perform?

In [ ]:
# TODO - fill in these values
average_precision = 0.0
average_recall = 0.0
average_fscore = 0.0

# What attributes had the highest F-scores?
best_attributes = {
    "attrbute_name1" : 0.0,
    "attrbute_name2" : 0.0,
    "attrbute_name3" : 0.0,
}

# What attributes had the lowest F-scores?
worst_attributes = {
    "attrbute_name1" : 0.0,
    "attrbute_name2" : 0.0,
    "attrbute_name3" : 0.0,
}

# What could you try to improve the model's performance?
# (e.g.: hyperparameters, prompts, choice of model to focus on finetuning, tasks to train on?)
potential_improvements = """
TODO
"""

# Finetuning via OpenAI API?

For the last few years, OpenAI (among other providers) offered online model finetuning services.

However, OpenAI at least seems to be deprecating this service. If you are curious how this worked before, you can read their [guide to model optimization](https://developers.openai.com/api/docs/guides/model-optimization) with examples of the relevant API calls.

# BONUS: Using and training a CPU-only model

We shared some cookbook code for (very slowly) performing inference using Gemma-2-2B-it on a CPU and training with a GPU. However, there are smaller LLMs designed to be relatively lightweight while still taking advantage of modern LLM work!

[SmolLM2-360M](https://huggingface.co/HuggingFaceTB/SmolLM2-360M) and its [Instruct variant](https://huggingface.co/HuggingFaceTB/SmolLM2-360M-Instruct) are two such examples.

Feel free to play around with these models and observe how their memory usage, inference speed, performance, and finetuneability compare against Gemma.

In [ ]:
# TODO OPTIONAL: Load models

In [ ]:
# TODO OPTIONAL: Perform inference with models, testing out any of our few-shot or zero-shot tasks from earlier

In [ ]:
# TODO OPTIONAL: How does the next-token probability distribution compare with Gemma?

In [ ]:
# TODO OPTIONAL: Can we finetune SmolLM on our wiki-box generation task, and parse its output? How does its performance compare?

# Feedback questions

In [ ]:
# Approximately how many hours did you spend on this assignment?
# Just an approximation is fine.
num_hours_spent = 5

# Which sections did you find the most interesting? Most challenging? Would you change anything?
# This assignment has undergone multiple major revisions, so your feedback is valuable.
feedback = """
The fine-tuning section was easily the most interesting part for me. I had used
ChatGPT-style tools plenty of times before, but I never actually understood what
was happening under the hood until I had to load the weights, quantize the model,
and watch the training loss drop over each step. Seeing the box-to-bio model
improve from the base Gemma output made the whole idea of fine-tuning click in a
way that lecture slides never did.

The most challenging part was the reverse bio-to-box task. Getting the model to
output attributes in a clean, parseable format was harder than I expected, and I
spent a while realizing that most of my low scores were coming from formatting
mismatches rather than the model actually being wrong. The evaluation step was
also slow, so debugging parse_bio on a single example first saved me a lot of
time.

If I would change anything, it would be adding a rough time estimate for each
individual section rather than just the whole assignment, since the GPU sections
took much longer than the earlier ones and it was hard to plan around that.
"""

# Did you collaborate with any humans or use AI tool assistance?
# What did you learn or borrow from online web searches, chat tools, etc?
reflections = """

I used Claude to help me when I was stuck with this homework
not by asking for the direct answer but ask me questions to
prompt me to develop and write the answer on my own. Using an AI
tool sped up my thinking process but I practiced tracing
through edge cases to make sure I understood what each function does.

I made sure to read through the resulting code and
reasoning so that I could still explain everything.
"""